# MedTrack_DV — KPI Calculation & Validation

## Milestone 2: KPI Engineering

This notebook calculates and validates the six mandatory KPIs for the MedTrack_DV Hospital Operations & Patient Analytics Dashboard.

### KPIs
1. Total Admissions
2. Occupancy Rate
3. Average Length of Stay
4. Readmission Rate
5. Bed Utilization Rate
6. Department Efficiency Score

The calculations use the processed datasets created during Milestone 1.

The four datasets have different grains, so they are not blindly merged.

In [1]:
import pandas as pd
from pathlib import Path

# Path to processed datasets
data_path = Path("../data/processed")

# Load datasets
hospital = pd.read_csv(data_path / "hospital_overview_dataset.csv")
flow = pd.read_csv(data_path / "patient_flow_dataset.csv")
department = pd.read_csv(data_path / "department_analytics_dataset.csv")
resource = pd.read_csv(data_path / "resource_utilization_dataset.csv")

print("Datasets loaded successfully.")
print("Hospital Overview:", hospital.shape)
print("Patient Flow:", flow.shape)
print("Department Analytics:", department.shape)
print("Resource Utilization:", resource.shape)

Datasets loaded successfully.
Hospital Overview: (460, 24)
Patient Flow: (1126, 20)
Department Analytics: (2154, 24)
Resource Utilization: (6462, 24)


In [2]:
# 1. Total Admissions
total_admissions = hospital["admission_id"].nunique()

# 2. Occupancy Rate
occupancy_rate = (
    department["occupied_beds_count"].sum()
    / department["total_beds"].sum()
) * 100

# 3. Average Length of Stay
average_los = hospital["length_of_stay_days"].mean()

# 4. Readmission Rate
readmission_rate = (
    hospital["readmission_flag"].eq("Yes").sum()
    / len(hospital)
) * 100

# 5. Bed Utilization Rate
beds = resource[
    resource["resource_type"].str.strip().str.casefold() == "bed"
]

bed_utilization_rate = (
    beds["units_in_use"].sum()
    / beds["total_units_available"].sum()
) * 100

# 6. Department Efficiency Score
department_efficiency_score = department["department_efficiency_score"].mean()

# Display KPI results
kpis = pd.DataFrame({
    "KPI": [
        "Total Admissions",
        "Occupancy Rate (%)",
        "Average Length of Stay",
        "Readmission Rate (%)",
        "Bed Utilization Rate (%)",
        "Department Efficiency Score"
    ],
    "Value": [
        total_admissions,
        occupancy_rate,
        average_los,
        readmission_rate,
        bed_utilization_rate,
        department_efficiency_score
    ]
})

kpis

,KPI,Value
0,Total Admissions,460.000000
1,Occupancy Rate (%),15.682117
2,Average Length of Stay,3.752609
3,Readmission Rate (%),5.217391
4,Bed Utilization Rate (%),15.682117
5,Department Efficiency Score,65.287187


In [3]:
# KPI validation

assert total_admissions > 0
assert 0 <= occupancy_rate <= 100
assert average_los >= 0
assert 0 <= readmission_rate <= 100
assert 0 <= bed_utilization_rate <= 100
assert 0 <= department_efficiency_score <= 100

print("KPI validation passed.")
print()
print(kpis)

KPI validation passed.

                           KPI       Value
0             Total Admissions  460.000000
1           Occupancy Rate (%)   15.682117
2       Average Length of Stay    3.752609
3         Readmission Rate (%)    5.217391
4     Bed Utilization Rate (%)   15.682117
5  Department Efficiency Score   65.287187


In [4]:
# Save Tableau-ready KPI workbook

output_path = data_path / "hospital_final_dataset.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    kpis.to_excel(writer, sheet_name="KPI_Summary", index=False)
    hospital.to_excel(writer, sheet_name="Hospital_Overview", index=False)
    flow.to_excel(writer, sheet_name="Patient_Flow", index=False)
    department.to_excel(writer, sheet_name="Department_Analytics", index=False)
    resource.to_excel(writer, sheet_name="Resource_Utilization", index=False)

print(f"Final KPI workbook saved to: {output_path}")

Final KPI workbook saved to: ..\data\processed\hospital_final_dataset.xlsx
